# Look Eyrie forcing comparison

Both hourly forcing files use a lapse rate of 0.6 °C per 100 m for HRRR-AK air temperatures, a cooling of about 2.30 °C from the grid cell to the station. Met mixed uses station temperature, paired wind speed and direction, and quality-screened hourly precipitation when available. HRRR-AK supplies gaps and all relative humidity, shortwave and longwave radiation, and air pressure. The coverage table counts actual station substitutions, including hours when the two sources report equal values.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
default_site = repo.parents[1] / "models" / "run_snowpack" / "sites" / "lookeyrie"
site = Path(os.environ.get("LOOK_EYRIE_SNOWPACK_SITE", default_site)).expanduser()
input_dir = site / "input"
if not input_dir.is_dir():
    raise FileNotFoundError(f"Look Eyrie inputs are missing at {input_dir}. Set LOOK_EYRIE_SNOWPACK_SITE.")
runs = {
    "HRRR-AK": "hrrrak_lapserate_lookeyrie",
    "Met mixed": "met_hrrrak_lapserate_lookeyrie",
}
colors = {"Station": "#000000", "HRRR-AK": "#1764aa", "Met mixed": "#238343"}

def read_smet(path, columns=None):
    header = {}
    with path.open() as stream:
        for line in stream:
            if line.strip() == "[DATA]":
                break
            if "=" in line:
                key, value = line.split("=", 1)
                header[key.strip()] = value.strip()
        fields = header["fields"].split()
        data = pd.read_csv(stream, sep=r"\s+", names=fields,
                           usecols=None if columns is None else ["timestamp", *columns],
                           na_values=[float(header["nodata"])])
    data.index = pd.DatetimeIndex(pd.to_datetime(data.pop("timestamp"), utc=True), name="UTC")
    return data, header


In [ ]:
forcing = {label: read_smet(input_dir / run / f"{run}.smet")[0]
           for label, run in runs.items()}
index = forcing["HRRR-AK"].index
assert index.equals(forcing["Met mixed"].index)
observed = pd.read_csv(input_dir / "look_eyrie_meteo_observations.csv",
                       index_col="timestamp", parse_dates=True).reindex(index)

station = observed.rename(columns={"temp": "TA", "wind": "VW", "wind_direction": "DW",
                                   "precip_hourly": "PSUM"})[["TA", "VW", "DW", "PSUM"]]
used = station.notna()
used[["VW", "DW"]] = observed["wind_pair_used"].to_numpy()[:, None] & used[["VW", "DW"]]
fields = forcing["HRRR-AK"].columns
hours = used.sum().reindex(fields, fill_value=0)
display(pd.DataFrame({"Station hours used": hours, "Station share (%)": (100 * hours / len(index)).round(1)}))
print(f"{len(index):,} hourly records from {index[0]} to {index[-1]}")

In [ ]:
series = {label: frame.assign(TA=frame.TA - 273.15, RH=frame.RH * 100, P=frame.P / 100)
          for label, frame in forcing.items()}
units = {"TA": "Air temperature (°C)", "RH": "Relative humidity (%)",
         "VW": "Wind speed (m/s)", "ISWR": "Shortwave (W/m²)",
         "ILWR": "Longwave (W/m²)", "PSUM": "Precipitation (mm/day)", "P": "Pressure (hPa)"}
fig, axes = plt.subplots(len(units), 1, figsize=(13, 12), sharex=True, layout="constrained")
for ax, (field, unit) in zip(axes, units.items()):
    for label, frame in {**series, "Station": station}.items():
        if field not in frame:
            continue
        daily = frame[field].resample("D")
        values = daily.sum(min_count=24) if field == "PSUM" else daily.mean().where(daily.count() >= 18)
        ax.plot(values, color=colors[label], label=label,
                lw=1.2 if label == "HRRR-AK" else 0.8,
                ls="--" if label == "Met mixed" else "-", alpha=0.85)
    ax.set_ylabel(unit)
    ax.grid(alpha=0.2)
axes[0].legend()
axes[-1].set_xlabel("UTC date")
plt.show()

Daily means require at least 18 valid hourly values. Daily precipitation requires 24 valid hours. The wind direction plot shows hourly frequencies by compass sector, avoiding an arithmetic average across north.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
bins = np.arange(-22.5, 360, 45)
for label, frame in {**series, "Station": station}.items():
    angles = (frame.DW.dropna() + 22.5) % 360 - 22.5
    counts, edges = np.histogram(angles, bins=bins)
    ax.stairs(100 * counts / counts.sum(), edges, color=colors[label], label=label,
              linestyle="--" if label == "Met mixed" else "-")
ax.set(xticks=np.arange(0, 360, 45), xticklabels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"],
       ylabel="Share of valid hours (%)", xlabel="Wind direction")
ax.legend()
plt.show()